In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

ROOT_DIR = Path("datasets")

REGISTRY_METADATA_FILE = (
    ROOT_DIR
    / "constructed_datasets/registry_reference_cohort.csv"
)

REGISTRY_FEATURE_FILE = (
    ROOT_DIR
    / "preprocessing"
    / "registry"
    / "eligibility_features.csv"
)

CHIA_FEATURE_FILE = (
    ROOT_DIR
    / "preprocessing"
    / "chia"
    / "eligibility_features.csv"
)

LCT_FEATURE_FILE = (
    ROOT_DIR
    / "preprocessing"
    / "lct"
    / "eligibility_features.csv"
)


input_files = {
    "Registry metadata": REGISTRY_METADATA_FILE,
    "Registry features": REGISTRY_FEATURE_FILE,
    "CHIA features": CHIA_FEATURE_FILE,
    "LCT features": LCT_FEATURE_FILE,
}

for label, path in input_files.items():

    if not path.exists():
        raise RuntimeError(
            f"{label} file not found: {path}"
        )


registry_metadata = pd.read_csv(
    REGISTRY_METADATA_FILE,
    encoding="utf-8-sig",
    low_memory=False,
)

registry_df = pd.read_csv(
    REGISTRY_FEATURE_FILE,
    encoding="utf-8-sig",
    low_memory=False,
)

chia_df = pd.read_csv(
    CHIA_FEATURE_FILE,
    encoding="utf-8-sig",
    low_memory=False,
)

lct_df = pd.read_csv(
    LCT_FEATURE_FILE,
    encoding="utf-8-sig",
    low_memory=False,
)


print(
    f"Registry metadata: "
    f"{len(registry_metadata):,} rows, "
    f"{registry_metadata['nct_id'].nunique():,} unique NCT IDs"
)

print(
    f"Registry features: "
    f"{len(registry_df):,} rows, "
    f"{registry_df['nct_id'].nunique():,} unique NCT IDs"
)

print(
    f"CHIA features: "
    f"{len(chia_df):,} rows, "
    f"{chia_df['nct_id'].nunique():,} unique NCT IDs"
)

print(
    f"LCT features: "
    f"{len(lct_df):,} rows, "
    f"{lct_df['nct_id'].nunique():,} unique NCT IDs"
)

Registry metadata: 404,236 rows, 404,236 unique NCT IDs
Registry features: 404,236 rows, 404,236 unique NCT IDs
CHIA features: 1,000 rows, 1,000 unique NCT IDs
LCT features: 1,006 rows, 1,006 unique NCT IDs


In [2]:
analysis_datasets = {
    "Registry metadata": registry_metadata,
    "Registry features": registry_df,
    "CHIA features": chia_df,
    "LCT features": lct_df,
}


for label, data in analysis_datasets.items():

    if data["nct_id"].isna().any():
        raise RuntimeError(
            f"Missing NCT IDs found in {label}."
        )

    if data["nct_id"].duplicated().any():
        raise RuntimeError(
            f"Duplicate NCT IDs found in {label}."
        )

    if data["nct_id"].nunique() != len(data):
        raise RuntimeError(
            f"Unique NCT-ID count does not match "
            f"the number of records in {label}."
        )


registry_metadata_ids = set(
    registry_metadata["nct_id"]
)

registry_feature_ids = set(
    registry_df["nct_id"]
)

if registry_metadata_ids != registry_feature_ids:

    metadata_only = (
        registry_metadata_ids
        - registry_feature_ids
    )

    features_only = (
        registry_feature_ids
        - registry_metadata_ids
    )

    raise RuntimeError(
        "Registry metadata and preprocessing feature "
        "datasets do not contain the same NCT IDs.\n"
        f"Metadata only: {len(metadata_only):,}\n"
        f"Features only: {len(features_only):,}"
    )


print("Analysis input integrity checks passed.")

for label, data in analysis_datasets.items():
    print(
        f"{label}: {len(data):,} studies"
    )

Analysis input integrity checks passed.
Registry metadata: 404,236 studies
Registry features: 404,236 studies
CHIA features: 1,000 studies
LCT features: 1,006 studies


In [3]:
def summarize_categorical(
    data,
    variable,
):
    """
    Summarize a categorical variable.

    Missing values are retained as a separate category.
    Percentages use all records in the supplied dataset
    as the denominator.
    """

    summary = (
        data[variable]
        .fillna("MISSING")
        .value_counts(dropna=False)
        .rename_axis("category")
        .reset_index(name="n_studies")
    )

    summary["percent_of_cohort"] = (
        summary["n_studies"]
        / len(data)
        * 100
    )

    return summary

In [4]:
study_status_summary = summarize_categorical(
    registry_metadata,
    "study_status",
)

phase_summary = summarize_categorical(
    registry_metadata,
    "phase",
)

sex_summary = summarize_categorical(
    registry_metadata,
    "sex",
)


for summary_name, summary_df in [
    ("study status", study_status_summary),
    ("phase", phase_summary),
    ("sex", sex_summary),
]:
    if summary_df["n_studies"].sum() != len(registry_metadata):
        raise RuntimeError(
            f"{summary_name.capitalize()} counts do not "
            "reconcile with the registry reference cohort."
        )

In [5]:
enrollment_numeric = pd.to_numeric(
    registry_metadata["enrollment"],
    errors="coerce",
)

enrollment_summary = pd.DataFrame(
    {
        "n_available": [
            int(enrollment_numeric.notna().sum())
        ],
        "n_missing": [
            int(enrollment_numeric.isna().sum())
        ],
        "median": [
            enrollment_numeric.median()
        ],
        "q1": [
            enrollment_numeric.quantile(0.25)
        ],
        "q3": [
            enrollment_numeric.quantile(0.75)
        ],
        "min": [
            enrollment_numeric.min()
        ],
        "max": [
            enrollment_numeric.max()
        ],
    }
)

enrollment_summary

,n_available,n_missing,median,q1,q3,min,max
0,403600,636,60.0,27.0,138.0,0.0,99999999.0


In [6]:
country_scope = np.select(
    [
        registry_metadata["n_countries"] == 0,
        registry_metadata["n_countries"] == 1,
        registry_metadata["n_countries"] > 1,
    ],
    [
        "No country information",
        "Single-country",
        "Multicountry",
    ],
    default="Unknown",
)

country_scope_summary = (
    pd.Series(
        country_scope,
        name="country_scope",
    )
    .value_counts()
    .rename_axis("country_scope")
    .reset_index(name="n_studies")
)

country_scope_summary["percent_of_cohort"] = (
    country_scope_summary["n_studies"]
    / len(registry_metadata)
    * 100
)

if (
    country_scope_summary["n_studies"].sum()
    != len(registry_metadata)
):
    raise RuntimeError(
        "Country-scope counts do not reconcile "
        "with the registry reference cohort."
    )

country_scope_summary

,country_scope,n_studies,percent_of_cohort
0,Single-country,339358,83.950465
1,Multicountry,32733,8.097498
2,No country information,32145,7.952038


In [7]:
print("Study status")
print(
    study_status_summary.to_string(
        index=False,
        formatters={
            "percent_of_cohort": "{:.1f}".format,
        },
    )
)

print()
print("Phase")
print(
    phase_summary.to_string(
        index=False,
        formatters={
            "percent_of_cohort": "{:.1f}".format,
        },
    )
)

print()
print("Sex")
print(
    sex_summary.to_string(
        index=False,
        formatters={
            "percent_of_cohort": "{:.1f}".format,
        },
    )
)

print()
print("Enrollment")
print(
    enrollment_summary.to_string(
        index=False,
    )
)

print()
print("Country scope")
print(
    country_scope_summary.to_string(
        index=False,
        formatters={
            "percent_of_cohort": "{:.1f}".format,
        },
    )
)

Study status
               category  n_studies percent_of_cohort
              COMPLETED     232360              57.5
                UNKNOWN      64044              15.8
             RECRUITING      38290               9.5
             TERMINATED      27591               6.8
  ACTIVE_NOT_RECRUITING      16316               4.0
              WITHDRAWN      12655               3.1
     NOT_YET_RECRUITING       8958               2.2
ENROLLING_BY_INVITATION       2748               0.7
              SUSPENDED       1274               0.3

Phase
     category  n_studies percent_of_cohort
      MISSING     214028              52.9
       PHASE2      54934              13.6
       PHASE1      42699              10.6
       PHASE3      34593               8.6
       PHASE4      30672               7.6
PHASE1|PHASE2      14804               3.7
PHASE2|PHASE3       6633               1.6
 EARLY_PHASE1       5873               1.5

Sex
category  n_studies percent_of_cohort
     ALL     347072 

In [8]:
numeric_complexity_metrics = {
    "n_total_listed_items": "Listed eligibility items",
    "total_eligibility_word_count": "Eligibility word count",
    "mean_sentence_length": "Mean sentence length",
    "flesch_reading_ease": "Flesch Reading Ease",
}

expression_complexity_metrics = {
    "quantitative_present": "Quantitative expressions",
    "temporal_present": "Temporal expressions",
    "negation_present": "Negation",
    "conditional_present": "Conditional expressions",
    "exception_present": "Exception expressions",
}

comparison_datasets = {
    "Registry": registry_df,
    "CHIA": chia_df,
    "LCT": lct_df,
}

In [9]:
numeric_summary_rows = []

for dataset_name, data in comparison_datasets.items():

    for variable, label in numeric_complexity_metrics.items():

        values = pd.to_numeric(
            data[variable],
            errors="coerce",
        )

        numeric_summary_rows.append(
            {
                "dataset": dataset_name,
                "variable": variable,
                "label": label,
                "n_available": int(values.notna().sum()),
                "n_missing": int(values.isna().sum()),
                "median": values.median(),
                "q1": values.quantile(0.25),
                "q3": values.quantile(0.75),
                "min": values.min(),
                "max": values.max(),
            }
        )


numeric_complexity_summary = pd.DataFrame(
    numeric_summary_rows
)


print(
    numeric_complexity_summary.to_string(
        index=False,
        formatters={
            "median": "{:.2f}".format,
            "q1": "{:.2f}".format,
            "q3": "{:.2f}".format,
            "min": "{:.2f}".format,
            "max": "{:.2f}".format,
        },
    )
)

 dataset                     variable                    label  n_available  n_missing median    q1     q3     min     max
Registry         n_total_listed_items Listed eligibility items       396988       7248  11.00  7.00  18.00    1.00  167.00
Registry total_eligibility_word_count   Eligibility word count       404236          0 126.00 66.00 279.00    1.00 2978.00
Registry         mean_sentence_length     Mean sentence length       404236          0  11.33  7.92  15.89    1.00  299.00
Registry          flesch_reading_ease      Flesch Reading Ease       404235          1  36.46 28.49  45.17 -301.27  121.73
    CHIA         n_total_listed_items Listed eligibility items          995          5  10.00  7.00  16.00    1.00   50.00
    CHIA total_eligibility_word_count   Eligibility word count         1000          0  88.00 58.00 201.25    5.00 1492.00
    CHIA         mean_sentence_length     Mean sentence length         1000          0   9.62  7.00  14.00    2.40   85.00
    CHIA        

In [10]:
numeric_table = (
    numeric_complexity_summary
    .assign(
        summary=lambda x: (
            x["median"].map("{:.2f}".format)
            + " ("
            + x["q1"].map("{:.2f}".format)
            + "–"
            + x["q3"].map("{:.2f}".format)
            + ")"
        )
    )
    .pivot(
        index="dataset",
        columns="label",
        values="summary",
    )
    .reindex(
        index=["Registry", "CHIA", "LCT"],
        columns=list(numeric_complexity_metrics.values()),
    )
    .reset_index()
)

numeric_table.columns.name = None

numeric_table

,dataset,Listed eligibility items,Eligibility word count,Mean sentence length,Flesch Reading Ease
0,Registry,11.00 (7.00–18.00),126.00 (66.00–279.00),11.33 (7.92–15.89),36.46 (28.49–45.17)
1,CHIA,10.00 (7.00–16.00),88.00 (58.00–201.25),9.62 (7.00–14.00),35.21 (26.40–44.70)
2,LCT,8.00 (5.00–11.00),74.00 (47.00–119.00),9.00 (6.69–12.33),38.22 (26.95–49.32)


In [11]:
expression_prevalence_rows = []

for dataset_name, data in comparison_datasets.items():

    for variable, label in expression_complexity_metrics.items():

        values = pd.to_numeric(
            data[variable],
            errors="coerce",
        )

        n_available = int(
            values.notna().sum()
        )

        n_missing = int(
            values.isna().sum()
        )

        n_present = int(
            values.sum()
        )

        prevalence_percent = (
            n_present / n_available * 100
            if n_available > 0
            else np.nan
        )

        expression_prevalence_rows.append(
            {
                "dataset": dataset_name,
                "variable": variable,
                "label": label,
                "n_available": n_available,
                "n_missing": n_missing,
                "n_present": n_present,
                "prevalence_percent": prevalence_percent,
            }
        )


expression_prevalence_summary = pd.DataFrame(
    expression_prevalence_rows
)


print(
    expression_prevalence_summary.to_string(
        index=False,
        formatters={
            "prevalence_percent": "{:.2f}".format,
        },
    )
)

 dataset             variable                    label  n_available  n_missing  n_present prevalence_percent
Registry quantitative_present Quantitative expressions       404236          0     353789              87.52
Registry     temporal_present     Temporal expressions       404236          0     227537              56.29
Registry     negation_present                 Negation       404236          0     233699              57.81
Registry  conditional_present  Conditional expressions       404236          0      86039              21.28
Registry    exception_present    Exception expressions       404236          0     105498              26.10
    CHIA quantitative_present Quantitative expressions         1000          0        889              88.90
    CHIA     temporal_present     Temporal expressions         1000          0        521              52.10
    CHIA     negation_present                 Negation         1000          0        469              46.90
    CHIA  condition

In [12]:
expression_table = (
    expression_prevalence_summary
    .assign(
        summary=lambda x: (
            x["n_present"].map("{:,}".format)
            + " ("
            + x["prevalence_percent"].map("{:.2f}".format)
            + "%)"
        )
    )
    .pivot(
        index="dataset",
        columns="label",
        values="summary",
    )
    .reindex(
        index=["Registry", "CHIA", "LCT"],
        columns=list(expression_complexity_metrics.values()),
    )
    .reset_index()
)

expression_table.columns.name = None

expression_table

,dataset,Quantitative expressions,Temporal expressions,Negation,Conditional expressions,Exception expressions
0,Registry,"353,789 (87.52%)","227,537 (56.29%)","233,699 (57.81%)","86,039 (21.28%)","105,498 (26.10%)"
1,CHIA,889 (88.90%),521 (52.10%),469 (46.90%),162 (16.20%),192 (19.20%)
2,LCT,787 (78.23%),352 (34.99%),449 (44.63%),69 (6.86%),97 (9.64%)
